# 03. Layer 3 — Orchestration（流程型多 Agent 協作）

**情境**：使用者說「我想去東京玩 3 天」，系統自動完成：

1. **機票 Agent** 找航班
2. **飯店 Agent** 找住宿
3. **行程 Agent** 整合上面結果寫成 3 天行程

順序固定、上一個的結果是下一個的輸入 — 這就是 **Orchestration（流程型）** 的典型模式。ADK 提供 `SequentialAgent` 直接實現這種 pipeline，不需要自己寫迴圈。

**對比的 Coordination（下一份 notebook）**：Agent 之間沒有固定順序，靠 LLM 自己決定下一步交給誰。

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import get_model, final_text

## 1. Mock 工具（航班 / 飯店）

示範用，用固定資料代替真實 API。實務上換成航空公司 API、Booking.com SDK 等。

In [2]:
from google.adk.tools import FunctionTool

def search_flights(origin: str, destination: str, depart_date: str) -> dict:
    """Search return flights between two cities on a given date.

    Args:
        origin: Origin city, e.g. 'Taipei'.
        destination: Destination city, e.g. 'Tokyo'.
        depart_date: Departure date in YYYY-MM-DD.
    """
    return {
        "flights": [
            {"airline": "長榮 BR-198", "depart": f"{depart_date} 08:30", "price_twd": 14800},
            {"airline": "星宇 JX-822", "depart": f"{depart_date} 13:10", "price_twd": 13200},
            {"airline": "全日空 NH-852", "depart": f"{depart_date} 18:00", "price_twd": 16500},
        ],
        "origin": origin,
        "destination": destination,
    }

def search_hotels(city: str, nights: int) -> dict:
    """Search hotels in a city for a given number of nights.

    Args:
        city: City name in English.
        nights: Number of nights.
    """
    return {
        "hotels": [
            {"name": "Shinjuku Granbell Hotel", "area": "新宿",  "price_twd_per_night": 4200},
            {"name": "Park Hotel Tokyo",       "area": "汐留",  "price_twd_per_night": 6800},
            {"name": "MIMARU Tokyo Ueno East", "area": "上野",  "price_twd_per_night": 5500},
        ],
        "city": city,
        "nights": nights,
    }

flight_tool = FunctionTool(func=search_flights)
hotel_tool = FunctionTool(func=search_hotels)

## 2. 三個 Sub-agent

**設計重點**：每個 sub-agent 都有 `output_key`，把自己的結果**寫進 session state**，後續的 agent 用 `{key}` 在 instruction 裡注入。這就是 ADK 內建的 agent 之間「傳資料」的標準做法。

In [3]:
from google.adk.agents import LlmAgent

flight_agent = LlmAgent(
    name="flight_agent",
    model=get_model(),
    instruction=(
        "你是機票搜尋專員。從使用者需求中抓出**起點城市**、**目的地**、**出發日期**，"
        "**呼叫 search_flights 工具**取得航班，再用條列格式列出 3 個選項（航班、出發時間、價格）。"
    ),
    tools=[flight_tool],
    output_key="flight_options",
)

hotel_agent = LlmAgent(
    name="hotel_agent",
    model=get_model(),
    instruction=(
        "你是飯店推薦專員。從使用者需求中抓出**目的地城市**跟**晚數**，"
        "**呼叫 search_hotels 工具**取得飯店，用條列格式列出 3 間（名稱、區域、每晚價格）。"
    ),
    tools=[hotel_tool],
    output_key="hotel_options",
)

itinerary_agent = LlmAgent(
    name="itinerary_agent",
    model=get_model(),
    instruction=(
        "你是行程規劃師。根據以下資訊，用繁體中文寫一份簡潔的 3 天東京行程：\n\n"
        "【機票選項】\n{flight_options}\n\n"
        "【飯店選項】\n{hotel_options}\n\n"
        "請推薦其中一個機票+飯店組合，並列出 Day 1 ~ Day 3 各兩個重點景點。"
    ),
    output_key="final_itinerary",
)

## 3. 組合：SequentialAgent 一行串起來

這就是 Orchestration 的精華 — **流程是寫死的、可預期的**，不需要 LLM 來決定誰先誰後。

In [4]:
from google.adk.agents import SequentialAgent

travel_pipeline = SequentialAgent(
    name="travel_pipeline",
    sub_agents=[flight_agent, hotel_agent, itinerary_agent],
    description="端到端旅遊規劃：機票 → 飯店 → 行程整合",
)
print(f"Pipeline 子 agent 數量：{len(travel_pipeline.sub_agents)}")
print("執行順序：", " → ".join(a.name for a in travel_pipeline.sub_agents))

Pipeline 子 agent 數量：3
執行順序： flight_agent → hotel_agent → itinerary_agent


In [5]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER, SID = "layer3_orch", "sean", "trip-1"
session_service = InMemorySessionService()
await session_service.create_session(app_name=APP, user_id=USER, session_id=SID)
runner = Runner(agent=travel_pipeline, app_name=APP, session_service=session_service)

request = "我想 2026-05-15 從台北出發去東京玩 3 天"
msg = types.Content(role="user", parts=[types.Part(text=request)])

print(f"使用者：{request}\n")
async for ev in runner.run_async(user_id=USER, session_id=SID, new_message=msg):
    if ev.is_final_response():
        text = final_text(ev)
        if text:
            print(f"\n🎯 [{ev.author}]\n{text}\n" + "─" * 40)

使用者：我想 2026-05-15 從台北出發去東京玩 3 天




🎯 [flight_agent]
以下為您在 **2026‑05‑15** 從 **台北** 飛往 **東京** 的三個可供選擇的航班：

- ✈️ **長榮航空 BR‑198** →　出發時間：08:30 |　票價：約 NT\$14,800  
- ✈️ **星宇航空 JX‑822** →　出發時間：13:10 |　票價：約 NT\$13,200  
- ✈️ **全日空 NH‑852**  →　出發時間：18:00 |　票價：約 NT\$16,500  

如需查詢回程或其他日期的資訊，隨時告訴我！祝旅途愉快 🎏
────────────────────────────────────────



🎯 [hotel_agent]
以下是在 **東京** 您 3 晚住宿期間可以考慮的 3 家評價不錯且性價比佳的酒店，每間皆以「每晚」價格標示（已折算成臺幣）：

| 酒店 | 所屬區域 | 每晚價格 |
|------|----------|-----------|
| **Shinjuku Granbell Hotel** | 新宿 | NT$ 4,200 |
| **Park Hotel Tokyo** | 汐留 | NT$ 6,800 |
| **MIMARU Tokyo Ueno East** | 上野 | NT$ 5,500 |

### 小提醒
* 若您需要更詳細資料（例如房型說明、取消政策、是否含早餐等），只要告訴我，我再幫您進一步查詢。
* 想要預訂或比較其它天數/不同價位，也都可以直接讓我知道喔！

祝您在東京有一段美好的旅行體驗 🌸✨
────────────────────────────────────────



🎯 [itinerary_agent]
## 📌 推薦組合  
**✈ 航班：⭐ 星宇航空 JX‑822** – 2026‑05‑15 13:10 起飛（約 NT$13,200）  
**🏨 入住：🗼 Shinjuke Granbell Hotel** – 位於新宿市中心、交通便利（NT$4,200／夜）

> **總花費概估**（不含餐飲與交通）：  
> - 機票：NT$13,200  
> - 住宿 3 夜： NT$4,200 × 3 ≒ **NT$12,600**  
> **≈ NT$25,800** 為本次 3 日＋2 夜（第一日晚到即入住）的基礎開支，是目前最具 CP 值的方案。

---

## 🗓 3 Days 東京精華行程 （每日兩大必訪景點）

| 日期 | 行前小提示 | 重點景點① | 重點景點② | 建議安排（上午 / 下午 / 晚上） |
|------|------------|---------|--------|------------------------------|
| **D1 – 5/15 抵達 / D2**<br>抵達關西國際机场(羽田)後<br>搭京急線 → 品川 → 山手線至 **新宿站**（約45 分鐘） <br>办理入住放下行李 | ★ 把握午后光線，適合拍街景；<br>★ 可先領取 Suica/Pasmo 電子卡 | **新宿御苑** <br>— 日本庭園＋法式、西式大草坪，可悠閑散步 | **東京都廳觀景台** <br>— 免费入场，到 45 層俯瞰整座城市，特別推薦黃昏時分 | 上午：新宿御苑賞櫻／春季楓葉（視當季而定）。<br>午後：返回酒店短暫休息 → 前往東京都廳觀景台。（若喜歡購物，可順路逛 *伊勢丹新宿本店*)<br>晚上：新宿歌舞伎町 “Omoide Yokocho” 小巷吃居酒屋燒烤，感受熱鬧夜生活。 |
| **D2 – 5/16** | ★ 搭山手線快速換乘，多條景點相連，節省通勤時間。 | **原宿・竹下通** <br>— 年輕潮流文化代表，甜點＆服飾滿分打卡地 | **表參道 森之丘│代官山** <br>— 高級精品店林立，同時也能走进古著咖啡館，享受慢活氛圍 | 上午：從新宿搭山手線直达原宿，探索竹下通與明治神宮外社會廣場。<br>下午：沿表參道漫歩至代官山，穿梭於設計品牌與文

In [6]:
# 看一下 session state，三個 agent 的輸出都在這
session = await session_service.get_session(app_name=APP, user_id=USER, session_id=SID)
print("=== Session State Keys ===")
for k in session.state:
    print(f"  {k}: {len(str(session.state[k]))} chars")

=== Session State Keys ===
  flight_options: 244 chars
  hotel_options: 367 chars
  final_itinerary: 2021 chars


## 結論 — Orchestration 的設計哲學

| 特性 | 描述 |
|------|------|
| **流程寫死** | `SequentialAgent` 的順序由開發者決定，不靠 LLM |
| **狀態傳遞** | 透過 `output_key` 寫入 session state，下游用 `{key}` 注入 |
| **可預測** | 同樣輸入 → 同樣執行路徑（只有內容因模型不同而異）|
| **適合場景** | 訂單流程、資料 ETL、報表生成、有 SLA 的任務 |

**ADK 的另外兩個 workflow agent**：
- `ParallelAgent` — 同時跑多個 sub-agent（適合「並行研究 + 最後彙整」）
- `LoopAgent` — 重複執行直到條件滿足（適合「寫稿 → 審稿 → 改稿」迭代）

下一站：`04_layer3_coordination.ipynb` — 把流程**鬆開**，讓 LLM 自己決定下一步。